# 🎙️➡️🎨 From Voice to Vision — Fase 7: Dall'Emozione all'Arte

Il gran finale *From Engineering to Arts*: la voce entra, la CNN ottimizzata riconosce
l'emozione, e le **probabilità predette** condizionano **Stable Diffusion** che dipinge
un'opera. Se la predizione è mista (es. sad 50% + fearful 40%), i prompt vengono **fusi**:
il quadro riflette la sfumatura emotiva, non solo la classe vincente.

Usiamo **SD-Turbo** (Stable Diffusion distillata: 2 step di denoising, ~2-3 s per immagine su T4).

In [ ]:
# === 1. Setup + modello SER ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm diffusers transformers accelerate

import numpy as np, torch, matplotlib.pyplot as plt
from src import config, data_loader, features
from src.models import cnn
data_loader.download_ravdess()
df = data_loader.build_index()

device = cnn.get_device()
ckpt = config.RESULTS_DIR / "best_cnn.pt"
assert ckpt.exists(), "best_cnn.pt mancante: caricalo su GitHub in results/ (vedi Fase 6)"
model, ck = cnn.load_checkpoint(ckpt, device)
print("Modello SER pronto:", ck["hp"])

In [ ]:
# === 2. Carica Stable Diffusion (SD-Turbo) — ~2 min la prima volta ===
from src.art import emotion_to_image as e2i
pipe = e2i.load_pipeline()
print("✓ Stable Diffusion pronto")

## 3) Galleria: una clip di test per ogni emozione → quadro
Per ciascuna delle 8 emozioni: prendiamo una clip del **test set** (attori mai visti),
la classifichiamo, costruiamo il prompt e generiamo l'opera.

In [ ]:
from IPython.display import Audio, display
test_df = df[df.split == "test"].reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(18, 9.5))
gallery = []
for i, emo in enumerate(config.EMOTIONS):
    row = test_df[test_df.emotion == emo].iloc[0]
    probs = e2i.predict_probs_from_wav(row.path, model, ck["norm"], device)
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED + i)
    gallery.append({"emo": emo, "desc": desc, "img": img, "path": row.path})
    ax = axes[i//4, i%4]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"reale: {emo}\npredetto: {desc}", fontsize=10)
plt.suptitle("From Voice to Vision — galleria delle emozioni (clip di test)", fontsize=15)
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "07_gallery.png", dpi=120)
plt.show()

## 4) Il caso 'misto': quando la voce è ambigua, il quadro fonde due emozioni

In [ ]:
# cerchiamo nel test una clip con predizione davvero mista (2ª emozione ≥ 25%)
found = None
for _, row in test_df.iterrows():
    probs = e2i.predict_probs_from_wav(row.path, model, ck["norm"], device)
    top2 = np.sort(probs)[::-1][:2]
    if top2[1] >= 0.25:
        found = (row, probs); break

if found:
    row, probs = found
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].bar(config.EMOTIONS, probs, color="#4C78A8")
    ax[0].set_title(f"probabilità (reale: {row.emotion})"); ax[0].tick_params(axis='x', rotation=40)
    ax[1].imshow(img); ax[1].axis("off"); ax[1].set_title(f"blend: {desc}", fontsize=11)
    plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "07_blend.png", dpi=120); plt.show()
    display(Audio(row.path))
    print("Prompt usato:\n", prompt)
else:
    print("Nessuna clip con predizione mista trovata (soglia 25%) — abbassa la soglia e riprova")

## 5) Demo live: la TUA voce → arte 🎤
Registra un breve audio (3 s) col telefono o il PC, salvalo come .wav e caricalo qui.
Perfetto da mostrare in sede d'esame.

In [ ]:
from google.colab import files as gfiles
up = gfiles.upload()   # carica un .wav con la tua voce
for fname in up:
    probs = e2i.predict_probs_from_wav(fname, model, ck["norm"], device)
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].bar(config.EMOTIONS, probs, color="#4C78A8")
    ax[0].set_title("emozione riconosciuta nella tua voce"); ax[0].tick_params(axis='x', rotation=40)
    ax[1].imshow(img); ax[1].axis("off"); ax[1].set_title(desc, fontsize=12)
    plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "07_my_voice.png", dpi=120); plt.show()
    display(Audio(fname))
    print("Prompt:", prompt)

### ✅ Fase 7 completata — la pipeline è COMPLETA! 🎉
Voce → feature MEL → CNN ottimizzata (bio-ispirata) → spiegazione (XAI) → **opera d'arte**.

**Mandami:** la galleria delle 8 emozioni, il caso blend e (se vuoi) la prova con la tua voce.
Poi ultima fase: **il report in stile paper + rifinitura della repo** per la consegna.